In [ ]:
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_score, recall_score
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-darkgrid")

## 15. Final Summary

Final model performance comparisons.

In [ ]:
path = Path("../data/processed/train_features.parquet")
if not path.exists():
    path = Path("../data/interim/train_merged.parquet")
train = pd.read_parquet(path)
print(f"Training data shape: {train.shape}")
print(f"Fraud rate: {train['isFraud'].mean()*100:.2f}%")

### 15.1 Final Model Performance

In [ ]:
exclude_cols = ["TransactionID", "isFraud", "TransactionDT"]
feature_cols = [c for c in train.select_dtypes(include="number").columns if c not in exclude_cols]
X = train[feature_cols]
y = train["isFraud"]
X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# LightGBM
dtrain_lgb = lgb.Dataset(X_tr, label=y_tr)
dval_lgb = lgb.Dataset(X_va, label=y_va, reference=dtrain_lgb)
model_lgb = lgb.train(
    {"objective": "binary", "metric": "auc", "num_leaves": 31, "learning_rate": 0.1, "verbose": -1, "random_state": 42},
    dtrain_lgb, num_boost_round=100, valid_sets=[dval_lgb], callbacks=[lgb.early_stopping(15, verbose=False)]
)
preds_lgb = model_lgb.predict(X_va, num_iteration=model_lgb.best_iteration)
auc_lgb = roc_auc_score(y_va, preds_lgb)
pr_lgb = average_precision_score(y_va, preds_lgb)

# XGBoost
dtrain_xgb = xgb.DMatrix(X_tr, label=y_tr)
dval_xgb = xgb.DMatrix(X_va, label=y_va)
model_xgb = xgb.train(
    {"objective": "binary:logistic", "eval_metric": "auc", "max_depth": 6, "learning_rate": 0.1, "verbosity": 0, "random_state": 42},
    dtrain_xgb, num_boost_round=100, evals=[(dval_xgb, "val")],
    callbacks=[xgb.callback.EarlyStopping(rounds=15, save_best=True, metric_name="auc", data_name="val")],
    verbose_eval=False
)
preds_xgb = model_xgb.predict(dval_xgb)
auc_xgb = roc_auc_score(y_va, preds_xgb)
pr_xgb = average_precision_score(y_va, preds_xgb)

# CatBoost
model_cb = CatBoostClassifier(iterations=100, learning_rate=0.1, depth=6, eval_metric="AUC", random_seed=42, verbose=False)
model_cb.fit(X_tr, y_tr, eval_set=(X_va, y_va), early_stopping_rounds=15)
preds_cb = model_cb.predict_proba(X_va)[:, 1]
auc_cb = roc_auc_score(y_va, preds_cb)
pr_cb = average_precision_score(y_va, preds_cb)

comparison = pd.DataFrame([
    {"model": "LightGBM", "Val ROC-AUC": auc_lgb, "Val PR-AUC": pr_lgb},
    {"model": "XGBoost", "Val ROC-AUC": auc_xgb, "Val PR-AUC": pr_xgb},
    {"model": "CatBoost", "Val ROC-AUC": auc_cb, "Val PR-AUC": pr_cb},
])
print(comparison.sort_values("Val ROC-AUC", ascending=False).to_string(index=False))

### 15.2 Best Model

In [ ]:
best_idx = comparison["Val ROC-AUC"].idxmax()
best_row = comparison.loc[best_idx]
print(f"Best Model: {best_row['model'].upper()}")
print(f"  ROC-AUC  : {best_row['Val ROC-AUC']:.5f}")
print(f"  PR-AUC   : {best_row['Val PR-AUC']:.5f}")